In [ ]:
import os
import sys
import urllib.request
import datetime
import time
import json

client_id = 'Vkvy1ZApAuhh3uFGt5NU'
client_secret = 'I9K3zRYoRJ'


#[CODE 1]
def getRequestUrl(url):    
    req = urllib.request.Request(url)
    req.add_header("X-Naver-Client-Id", client_id)
    req.add_header("X-Naver-Client-Secret", client_secret)
    
    try: 
        response = urllib.request.urlopen(req)
        if response.getcode() == 200:
            print ("[%s] Url Request Success" % datetime.datetime.now())
            return response.read().decode('utf-8')
    except Exception as e:
        print(e)
        print("[%s] Error for URL : %s" % (datetime.datetime.now(), url))
        return None

#[CODE 2]
def getNaverSearch(node, srcText, start, display):    
    base = "https://openapi.naver.com/v1/search"
    node = "/%s.json" % node
    #                                                    url 인코딩(한글)
    parameters = "?query=%s&start=%s&display=%s" % (urllib.parse.quote(srcText), start, display)
    
    url = base + node + parameters    
    responseDecode = getRequestUrl(url)   #[CODE 1]
    
    if (responseDecode == None):
        return None
    else:
        return json.loads(responseDecode) # 문자열JSON->PY객체로 변환
                                        #{}->dict, []->list

#[CODE 3]
def getPostData(post, jsonResult, cnt):    
    title = post['title']
    description = post['description']
    org_link = post['originallink']
    link = post['link']
    #strptime():문자열->날짜
    pDate = datetime.datetime.strptime(post['pubDate'],  '%a, %d %b %Y %H:%M:%S +0900')
    #strftime():날짜->문자열
    pDate = pDate.strftime('%Y-%m-%d %H:%M:%S')
    
    jsonResult.append({'cnt':cnt, 'title':title, 'description': description, 
'org_link':org_link,   'link': link,   'pDate':pDate})
    return    

#[CODE 0]
def main():
    node = 'news'   # 크롤링 할 대상
    srcText = input('검색어를 입력하세요: ')
    cnt = 0
    jsonResult = []

    jsonResponse = getNaverSearch(node, srcText, 1, 100)  #[CODE 2]
    total = jsonResponse['total']
 
    while ((jsonResponse != None) and (jsonResponse['display'] != 0)):         
        for post in jsonResponse['items']:#배열
            cnt += 1
            getPostData(post, jsonResult, cnt)  #[CODE 3]       
        
        start = jsonResponse['start'] + jsonResponse['display'] # start값 재설정
        jsonResponse = getNaverSearch(node, srcText, start, 100)  #[CODE 2]
       
    print('전체 검색 : %d 건' %total)
    
    with open('%s_naver_%s.json' % (srcText, node), 'w', encoding='utf8') as outfile:
        #json.dump():py객체를 json문자열로 생성해줌. indent(들여쓰기)로 가독성 향상
        #sort_keys=True 키를 알파벳 순으로 정렬
        #ensure_ascii=False:한글 그대로 출력(없으면 유니코드로 출력)
        jsonFile = json.dumps(jsonResult,  indent=4, sort_keys=True,  ensure_ascii=False)
                        
        outfile.write(jsonFile)
        
    print("가져온 데이터 : %d 건" %(cnt))
    print ('%s_naver_%s.json SAVED' % (srcText, node))
    
if __name__ == '__main__':  #임포트되지 않고 직접 실행하는 경우라면
    main()                  #main 함수 호출


[2026-03-27 12:05:35.549152] Url Request Success
[2026-03-27 12:05:35.700008] Url Request Success
[2026-03-27 12:05:35.824373] Url Request Success
[2026-03-27 12:05:35.948604] Url Request Success
[2026-03-27 12:05:36.076339] Url Request Success
[2026-03-27 12:05:36.212810] Url Request Success
[2026-03-27 12:05:36.440781] Url Request Success
[2026-03-27 12:05:36.598833] Url Request Success
[2026-03-27 12:05:36.769748] Url Request Success
[2026-03-27 12:05:36.935398] Url Request Success
HTTP Error 400: Bad Request
[2026-03-27 12:05:36.988035] Error for URL : https://openapi.naver.com/v1/search/news.json?query=%EC%9D%B4%EB%9E%80&start=1001&display=100
전체 검색 : 1135433 건
가져온 데이터 : 1000 건
이란_naver_news.json SAVED
